# M04-01 — Joins

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-kpis.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Medir cuántas líneas y pedidos se pierden al hacer inner contra clientes, y listar los huérfanos.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M04-01-joins.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras. No es adorno: es la traza de tu razonamiento.
2. **Código** — lo pegas o lo escribes, lo **ejecutas** (`Shift+Enter`), **miras** la salida y, si no cuadra, lo **mejoras**.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Carga fact y clientes

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> El fact ya trae customer_id de la cabecera. Leo Parquet, no CSV.

**2. Crea una celda de código** debajo y escribe:

```python
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
print(fact.count(), customers.count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** `1980 250`.

**Por qué este paso.** Si falta fact_lines, cierra M03-02 primero.


### Paso 2 — Inner frente a left

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> La diferencia ES el síntoma de las claves huérfanas. Cuento los dos.

**2. Crea una celda de código** debajo y escribe:

```python
inner = fact.join(customers, "customer_id", "inner")
left = fact.join(customers, "customer_id", "left")
print("inner", inner.count(), "left", left.count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** inner **1956** · left **1980**.

**Por qué este paso.** Si el inner sale mayor que 1980, el join de catálogo te ha duplicado (no lo hagas aquí).


### Paso 3 — Anti-join de huérfanos

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> left_anti = está en el fact y no en clientes. Mejor que un where a ciegas.

**2. Crea una celda de código** debajo y escribe:

```python
orphans = fact.join(customers, "customer_id", "left_anti")
orphans.select("order_id", "customer_id").distinct().orderBy("order_id").show()
print("líneas", orphans.count(), "pedidos", orphans.select("order_id").distinct().count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** **24** líneas · **8** pedidos · `customer_id` tipo `CX*`.

**Por qué este paso.** Si no ves CX*, los filtraste en M02-03: regenera staging.

Opcional: left a `products_clean` por `product_id`. Las líneas `P999` aparecen con `name` nulo: mismo patrón.


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

`fact.count() - inner.count()` → **24** líneas (8 pedidos). Anótalo en Markdown.


## Mejora — Mismo patrón a grano pedido

Inner/left de `orders_clean` ⋈ `customers_clean`. Markdown que compare con el grano línea.

<details>
<summary>Si te atascas, mira una solución</summary>

```python
orders = spark.read.parquet(str(STAGING / "orders_clean"))
print("orders", orders.count())
print("inner", orders.join(customers, "customer_id", "inner").count())  # 780
print("left ", orders.join(customers, "customer_id", "left").count())   # 788
```

</details>


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Inner > 1980 | Productos duplicados en otro join | `dropDuplicates` en product_id |
| No veo CX* | Los filtraste en M02-03 | Regenera staging: solo quitas customer_id vacío |
| customer_id ambiguo | Join mal nombrado | Usa `join(..., "customer_id")` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M04-02 KPIs](03-lab-kpis.ipynb).
